In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *


In [2]:
def remap(data, header, nx=720, ny=1800, **kwargs):
    from transforms import ToSpherical
    from scipy.ndimage import map_coordinates

    view = View.from_header(header).update(**kwargs)
    transform = ~ToSpherical() - view.to_carrington(**kwargs)

    lat_spread = 2
    dlon = 360 / ny
    dlat = lat_spread / (nx - 1)


    grid = np.mgrid[-lat_spread / 2:lat_spread / 2 + dlat / 2:dlat, :360:dlon]
    grid[0] = np.arcsin(grid[0].clip(-1,1)) * 180 / np.pi

    grid, alpha = transform(grid)
    data_ = map_coordinates(np.nan_to_num(data), grid, cval=np.nan)

    return data_, grid, alpha


In [3]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2315/*')))

files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/sdo/hmi/2315/hmi.M_720s.20260830_000000_TAI.3.magnetogram.fits'),
 np.str_('/home/ulyanov/data/sdo/hmi/2315/hmi.M_720s.20260926_000000_TAI.3.magnetogram.fits'),
 107)

In [4]:
N = 0

LL = 0
GG = 0
LG = 0

LF = 0
GF = 0

FF = 0


for file in files[:]:

    with fits.open(file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    data = rebin(data, 4, update_header=header)

    data_, grid, alpha = remap(data, header, delta_t=20, correct_dr=True, minmu=0.1)

    x, y = grid
    x, y = (x - 512) / 512, (y - 512) / 512

    z = 1 / alpha
    n = (~np.isnan(data_)).astype(float)


    l = np.expand_dims(z, (-2,-1))
    #g = np.moveaxis(np.array([[x ** 3, x ** 2 * y, x * y ** 2, y ** 3, x ** 2, x * y, y ** 2, x, y, n]]), (0,1), (-2,-1))
    #g = np.moveaxis(np.array([[x ** 2, x * y, y ** 2, x, y, n]]), (0,1), (-2,-1))
    g = np.moveaxis(np.array([[x, y, n]]), (0,1), (-2,-1))
    #g = np.moveaxis(np.array([[n]]), (0,1), (-2,-1))
    f = np.expand_dims(data_, (-2,-1))

    ll = np.swapaxes(l, -2, -1) @ l
    gg = np.swapaxes(g, -2, -1) @ g
    lg = np.swapaxes(l, -2, -1) @ g

    lf = np.swapaxes(l, -2, -1) @ f
    gf = np.swapaxes(g, -2, -1) @ f
    ff = np.swapaxes(f, -2, -1) @ f


    N += np.nan_to_num(n)
    w = np.expand_dims(n / N, (-2, -1))

    LL += np.nan_to_num((ll - LL) * w)
    GG += np.nan_to_num((gg - GG) * w)
    LG += np.nan_to_num((lg - LG) * w)
    LF += np.nan_to_num((lf - LF) * w)
    GF += np.nan_to_num((gf - GF) * w)
    FF += np.nan_to_num((ff - FF) * w)


LL[N == 0] = np.nan
GG[N == 0] = np.nan
LG[N == 0] = np.nan

LF[N == 0] = np.nan
GF[N == 0] = np.nan
FF[N == 0] = np.nan

N[N == 0] = np.nan

/tmp/ipykernel_224672/414263703.py:47: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))


In [5]:
LLi = np.linalg.inv(LL)

Q = LLi @ LF
R = LLi @ LG
GL = np.swapaxes(LG, -2, -1)


S = (FF - GF[:,:,-1:,:] ** 2 - Q ** 2 * (LL - GL[:,:,-1:,:] ** 2)) / LL / np.expand_dims(N - 1, (-1,-2))

D = np.linalg.inv(np.nanmean((GG - GL @ R) / S, axis=(0,1)))
V = np.nanmean((GF - GL @ Q) / S, axis=(0,1))
P = D @ V
dQ = R @ P

Q_ = Q - dQ

S = np.squeeze(S)
P = np.squeeze(P)
Q = np.squeeze(Q)
Q_ = np.squeeze(Q_)
dQ = np.squeeze(dQ)

print(P)

[-0.01133774 -0.03043018 -0.00614688]


In [6]:
show_map(np.sqrt(S), vmin=0, vmax=5, sine_lat=True, cmap='inferno')

In [12]:
show_map(Q_, vmin=-500, vmax=500, sine_lat=True)

In [8]:
show_map(dQ, vmin=-5, vmax=5, sine_lat=True, cmap='seismic')

In [11]:
nx = len(Q)

lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

#flux = np.nanmean(Q * N ** 2, axis=1) / np.nanmean(N ** 2, axis=1)
#flux_ = np.nanmean(Q_ * N ** 2, axis=1) / np.nanmean(N ** 2, axis=1)

flux = np.nanmean(Q, axis=1)
flux_ = np.nanmean(Q_, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux)
plt.plot(lat, flux_)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_224672/941732730.py:8: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(Q, axis=1)
/tmp/ipykernel_224672/941732730.py:9: RuntimeWarning: Mean of empty slice
  flux_ = np.nanmean(Q_, axis=1)


In [13]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2315.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.09.12_09:28:02_TAI'

In [14]:
show_map(data_, vmin=-500, vmax=500, sine_lat=True)

In [17]:
#flux_hmi = np.nanmean(data_ * Z2, axis=1) / np.nanmean(Z2, axis=1)
flux_hmi = np.nanmean(data_, axis=1)

nx, ny = data_.shape
lat_hmi = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_hmi, flux_hmi, label='HMI')
plt.plot(lat, flux, label='HMIav')
plt.plot(lat, flux_, label='HMIav (corrected)', color='red')

plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_224672/2349116668.py:2: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_, axis=1)


In [20]:
'fdt' in files[0]

False